# Sesión 15 sep 2026 — Pandas y procesamiento de datos

**Formato:** 30 min exposición (esta guía + live) · 30 min ejercicios (`ejercicios/E1_pandas.md`) · resto Proyecto I.

Guía docente: `../sesiones/2026-09-15_pandas_proyecto_i.md`.

## Objetivos
1. Diagnosticar un CSV (tipos, nulos, basura).
2. Validar con reglas de negocio y separar errores.
3. Agregar y exportar con `pathlib`.

## 1) Carga y diagnóstico
Siempre parte de un `Path` reproducible y de un vistazo a calidad.

In [ ]:
from pathlib import Path
import json
import pandas as pd

DATA = Path("Datos") / "ventas.csv"
df = pd.read_csv(DATA)

print("shape:", df.shape)
print("\ndtypes:\n", df.dtypes)
print("\nnulos:\n", df.isna().sum())
df


## 2) Tipado seguro
`to_numeric(..., errors="coerce")` convierte basura en `NaN` en lugar de petar.

In [ ]:
work = df.copy()
work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")
work[["unidades", "precio_unitario"]]


## 3) Validación como contrato
No borres filas en silencio: **devuelve** válidos y errores.

In [ ]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Separa filas válidas e inválidas según reglas de negocio."""
    work = frame.copy()
    work["unidades"] = pd.to_numeric(work["unidades"], errors="coerce")
    work["precio_unitario"] = pd.to_numeric(work["precio_unitario"], errors="coerce")
    mask_ok = (
        work["unidades"].notna()
        & (work["unidades"] > 0)
        & work["precio_unitario"].notna()
        & (work["precio_unitario"] > 0)
    )
    validos = work.loc[mask_ok].copy()
    errores = work.loc[~mask_ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores

validos, errores = validar_ventas(df)
print(f"válidos={len(validos)} errores={len(errores)}")
errores


## 4) Agregaciones de negocio

In [ ]:
por_region = (
    validos.groupby("region", as_index=False)["importe"].sum()
    .sort_values("importe", ascending=False)
)
top_productos = (
    validos.groupby("producto", as_index=False)["importe"].sum()
    .sort_values("importe", ascending=False)
    .head(3)
)
clientes_recurrentes = (
    validos.groupby("cliente_id").size().reset_index(name="compras")
    .query("compras > 1")
)
por_region, top_productos, clientes_recurrentes


## 5) Exportación + informe de calidad
Este informe es el embrión de las *metrics* de un pipeline.

In [ ]:
out_csv = Path("Datos") / "ventas_limpias.csv"
out_json = Path("Datos") / "calidad_datos.json"
validos.to_csv(out_csv, index=False)
informe = {
    "filas_totales": int(len(df)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
out_json.write_text(json.dumps(informe, indent=2, ensure_ascii=False), encoding="utf-8")
informe


## 6) Mini-retos para la exposición (preguntas al aula)
- ¿Qué regla añadirías para `cliente_id`?
- ¿Cómo detectarías duplicados exactos de factura?
- ¿Qué rompería si mañana llega una columna `descuento`?

## Siguiente bloque (30 min)
Abre `ejercicios/E1_pandas.md` y sigue los checkpoints cronometrados.